# YOLOv8 Test Prediction FIX — Doğru Sıralı NPY Üretimi

## Sorun
YOLOv8 notebook'u test prediction'ı **sınıf-gruplu** yapıyor (önce tüm akiec, sonra tüm bcc, ...).
EfficientNet ve Mask R-CNN ise pandas index sırasıyla işliyor (image_id sırası).
Bu yüzden 3 modelin `test_labels_*.npy` dosyaları farklı sırada oluyor → ensemble bozuluyor.

## Çözüm
Mevcut `best_yolov8n_cls.pt` modelini kullanıp inference'ı **pandas test_df sırasında** yeniden yap.
Hiçbir yeniden eğitim YOK. ~5-7 dakika sürer.

## Sonuç
`probabilities_yolov8n_cls.npy` ve `test_labels_yolov8n_cls.npy` Drive'da güncellenir.
Sonra ensemble notebook'u sorunsuz çalışır.

In [ ]:
# ============================================================
# A. GOOGLE DRIVE'I MOUNT ET
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# B. KAGGLE.JSON YÜKLE (HAM10000 indirmek için)
# ============================================================
from google.colab import files
print('Lütfen bilgisayarından kaggle.json dosyasını seç:')
uploaded = files.upload()

In [ ]:
# ============================================================
# C. HAM10000 DATASET'İ İNDİR (ROI hesaplamak için lazım)
# ============================================================
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!mv kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

!pip install -q kaggle ultralytics

print('\n📥 HAM10000 indiriliyor (~5 GB, 1-2 dakika)...')
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/ham10000 --unzip

print('\n✅ İndirme tamamlandı')

In [ ]:
# ============================================================
# 1. KÜTÜPHANELER
# ============================================================
import os
import numpy as np
import pandas as pd
import cv2
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from ultralytics import YOLO
from pathlib import Path
import shutil
import time

print('Kütüphaneler yüklendi.')

In [ ]:
# ============================================================
# 2. AYARLAR (orijinal YOLOv8 notebook'u ile birebir aynı)
# ============================================================
IMG_SIZE   = 224
SEED       = 42
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.15

DATA_DIR     = '/content/ham10000'
METADATA_CSV = os.path.join(DATA_DIR, 'HAM10000_metadata.csv')
IMG_DIR_1    = os.path.join(DATA_DIR, 'HAM10000_images_part_1')
IMG_DIR_2    = os.path.join(DATA_DIR, 'HAM10000_images_part_2')

DRIVE_OUTPUT = '/content/drive/MyDrive/cilt_kanseri_ensemble'
MODEL_PATH   = os.path.join(DRIVE_OUTPUT, 'best_yolov8n_cls.pt')

TEST_ROI_DIR = '/content/yolo_test_roi_fix'  # Sadece test ROI'leri için geçici klasör

TARGET_CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Model path: {MODEL_PATH}')
print(f'Model var mı? {os.path.exists(MODEL_PATH)}')

In [ ]:
# ============================================================
# 3. METADATA + LESION-BASED SPLIT (orijinal ile birebir aynı seed)
# ============================================================
df = pd.read_csv(METADATA_CSV)

# Image path eşleştirme
def find_image_path(image_id):
    p1 = os.path.join(IMG_DIR_1, f'{image_id}.jpg')
    p2 = os.path.join(IMG_DIR_2, f'{image_id}.jpg')
    if os.path.exists(p1):
        return p1
    elif os.path.exists(p2):
        return p2
    return None

df['image_path'] = df['image_id'].apply(find_image_path)
df = df.dropna(subset=['image_path']).reset_index(drop=True)
df = df[df['dx'].isin(TARGET_CLASSES)].reset_index(drop=True)

# Lesion-based stratified split (EfficientNet/Mask ile birebir aynı)
lesion_df = df.groupby('lesion_id').first().reset_index()[['lesion_id', 'dx']]

trainval_lesions, test_lesions = train_test_split(
    lesion_df, test_size=TEST_SPLIT, stratify=lesion_df['dx'], random_state=SEED
)
val_ratio = VAL_SPLIT / (1 - TEST_SPLIT)
train_lesions, val_lesions = train_test_split(
    trainval_lesions, test_size=val_ratio, stratify=trainval_lesions['dx'], random_state=SEED
)

test_df = df[df['lesion_id'].isin(test_lesions['lesion_id'])].reset_index(drop=True)
print(f'Test set: {len(test_df)} görüntü')
print(f'\nTest sınıf dağılımı:')
print(test_df["dx"].value_counts())

In [ ]:
# ============================================================
# 4. ROI EXTRACT FONKSİYONU (orijinal notebook ile aynı)
# ============================================================
def extract_roi(img_path, target_size=224, padding_pct=0.15):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    kernel = np.ones((5, 5), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return cv2.resize(img_rgb, (target_size, target_size))
    largest = max(contours, key=cv2.contourArea)
    x, y, w, h = cv2.boundingRect(largest)
    pad_x = int(w * padding_pct)
    pad_y = int(h * padding_pct)
    x1 = max(0, x - pad_x)
    y1 = max(0, y - pad_y)
    x2 = min(img_rgb.shape[1], x + w + pad_x)
    y2 = min(img_rgb.shape[0], y + h + pad_y)
    roi = img_rgb[y1:y2, x1:x2]
    if roi.size == 0:
        return cv2.resize(img_rgb, (target_size, target_size))
    return cv2.resize(roi, (target_size, target_size))

print('extract_roi hazır')

In [ ]:
# ============================================================
# 5. TEST ROI'LERİNİ HAZIRLA (PANDAS INDEX SIRASINDA)
# ============================================================
# Düz bir klasöre, image_id.jpg olarak kaydediyoruz.
# Sonra inference yaparken test_df.iterrows() ile pandas sırasında okuyacağız.

if os.path.exists(TEST_ROI_DIR):
    shutil.rmtree(TEST_ROI_DIR)
os.makedirs(TEST_ROI_DIR, exist_ok=True)

print('Test ROI\'leri hazırlanıyor...')
start = time.time()
fail = 0
for idx, row in test_df.iterrows():
    roi = extract_roi(row['image_path'], IMG_SIZE)
    if roi is None:
        fail += 1
        continue
    out_path = os.path.join(TEST_ROI_DIR, f"{row['image_id']}.jpg")
    cv2.imwrite(out_path, cv2.cvtColor(roi, cv2.COLOR_RGB2BGR))

elapsed = time.time() - start
n_files = len(os.listdir(TEST_ROI_DIR))
print(f'\nTamamlandı: {n_files} ROI | Hata: {fail} | Süre: {elapsed:.1f}s')

In [ ]:
# ============================================================
# 6. PANDAS INDEX SIRASINDA INFERENCE
# ============================================================
# test_df.iterrows() sırasıyla her görüntü için prediction.
# Bu sayede y_true ve y_probs, EfficientNet/Mask ile aynı sırada olacak.

model = YOLO(MODEL_PATH)

# YOLOv8 sınıf sırası kontrol
yolo_class_names = [model.names[i] for i in range(len(model.names))]
print(f'YOLOv8 sınıf sırası: {yolo_class_names}')
print(f'Bizim sıramız:        {TARGET_CLASSES}')
assert yolo_class_names == TARGET_CLASSES, 'Sınıf sıraları uyumsuz!'
print('✅ Sınıf sıraları uyumlu')

# image_id'lerin path'lerini pandas sırasında topla
test_paths = []
test_classes = []
missing = 0
for idx, row in test_df.iterrows():
    p = os.path.join(TEST_ROI_DIR, f"{row['image_id']}.jpg")
    if os.path.exists(p):
        test_paths.append(p)
        test_classes.append(row['dx'])
    else:
        missing += 1

print(f'\nInference yapılacak görüntü: {len(test_paths)} | Eksik: {missing}')

# Sınıf adı → index map
cls_to_idx = {cls: i for i, cls in enumerate(TARGET_CLASSES)}
y_true = np.array([cls_to_idx[c] for c in test_classes])

# Batch inference (hızlı)
print('\nInference başlıyor...')
start = time.time()
BATCH = 64
y_probs_list = []
for i in range(0, len(test_paths), BATCH):
    batch_paths = test_paths[i:i+BATCH]
    results = model(batch_paths, imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    for r in results:
        probs = r.probs.data.cpu().numpy()
        y_probs_list.append(probs)

y_probs = np.array(y_probs_list)
elapsed = time.time() - start
print(f'\nInference tamamlandı: {elapsed:.1f}s')
print(f'y_probs shape: {y_probs.shape}')
print(f'y_true shape : {y_true.shape}')

In [ ]:
# ============================================================
# 7. DOĞRULAMA — yeni sonuçlar eski metriklere uyuyor mu?
# ============================================================
# Eski raporlanan: Acc=0.8029, F1=0.5930
# Permütasyon sıralı veriden bağımsızdır → aynı çıkmalı.

y_pred = np.argmax(y_probs, axis=1)
acc = accuracy_score(y_true, y_pred)
f1  = f1_score(y_true, y_pred, average='macro')

print(f'YENİ Test Accuracy : {acc:.4f}  (orijinal: 0.8029)')
print(f'YENİ Test F1 macro : {f1:.4f}  (orijinal: 0.5930)')
print()
if abs(acc - 0.8029) < 0.005 and abs(f1 - 0.5930) < 0.005:
    print('✅ Sonuçlar orijinaliyle eşleşiyor — fix doğru çalışıyor!')
else:
    print('⚠️ Sonuçlar farklı — bir şey ters gitmiş olabilir.')
    print('  Sınıf bazlı karşılaştırma:')
    print(classification_report(y_true, y_pred, target_names=TARGET_CLASSES, digits=4))

In [ ]:
# ============================================================
# 8. NPY DOSYALARINI DRIVE'A KAYDET (eskinin üzerine yaz)
# ============================================================
PROBS_OUT  = os.path.join(DRIVE_OUTPUT, 'probabilities_yolov8n_cls.npy')
LABELS_OUT = os.path.join(DRIVE_OUTPUT, 'test_labels_yolov8n_cls.npy')

np.save(PROBS_OUT,  y_probs)
np.save(LABELS_OUT, y_true)

print(f'Kaydedildi:')
print(f'  {PROBS_OUT}  | shape: {y_probs.shape}')
print(f'  {LABELS_OUT} | shape: {y_true.shape}')

In [ ]:
# ============================================================
# 9. SON KONTROL — YOLOv8 labels artık diğerleriyle aynı sırada mı?
# ============================================================
eff_labels  = np.load(os.path.join(DRIVE_OUTPUT, 'test_labels_efficientnet_b0.npy'))
mask_labels = np.load(os.path.join(DRIVE_OUTPUT, 'test_labels_maskrcnn.npy'))
yolo_labels = np.load(LABELS_OUT)

print('Labels eşitliği:')
print(f'  YOLOv8 == EfficientNet : {np.array_equal(yolo_labels, eff_labels)}')
print(f'  YOLOv8 == Mask R-CNN   : {np.array_equal(yolo_labels, mask_labels)}')
print(f'  EfficientNet == Mask   : {np.array_equal(eff_labels, mask_labels)}')

if np.array_equal(yolo_labels, eff_labels) and np.array_equal(yolo_labels, mask_labels):
    print('\n🎉 BAŞARILI! 3 modelin labels\'ı artık aynı sırada.')
    print('Ensemble notebook\'unu çalıştırabilirsin.')
else:
    print('\n⚠️ Hala farklı — daha derin bir tutarsızlık var.')